### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Step 2: Open the project directory

Replace `Your_Dir` with your own path.

In [2]:
!git clone https://github.com/Calvin-Pang/emg2qwerty.git
%cd emg2qwerty

Cloning into 'emg2qwerty'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 137 (delta 28), reused 15 (delta 15), pack-reused 62 (from 2)
Receiving objects: 100% (137/137), 33.51 MiB | 16.43 MiB/s, done.
Resolving deltas: 100% (29/29), done.
Filtering content: 100% (17/17), 1.00 GiB | 12.89 MiB/s, done.
/content/emg2qwerty


### Step 3: Install required packages

After installing them, Colab will require you to restart the session.

In [4]:
%%capture
!pip install -r requirements.txt

In [5]:
# downloading dataset
!mkdir -p /content/emg2qwerty/data
!cp /content/drive/MyDrive/emg2qwerty_data.zip /content/emg2qwerty/data/
!unzip /content/emg2qwerty/data/emg2qwerty_data.zip -d /content/emg2qwerty/data/
!mv /content/emg2qwerty/data/emg2qwerty_data/* /content/emg2qwerty/data/

# adding my changes to be transformer architecture into the codebase
!cp /content/drive/MyDrive/modules.py /content/emg2qwerty/emg2qwerty/modules.py
!cp /content/drive/MyDrive/lightning.py /content/emg2qwerty/emg2qwerty/lightning.py
!cp /content/drive/MyDrive/transformer.yaml /content/emg2qwerty/config/model/transformer.yaml

Archive:  /content/emg2qwerty/data/emg2qwerty_data.zip
   creating: /content/emg2qwerty/data/emg2qwerty_data/
  inflating: /content/emg2qwerty/data/__MACOSX/._emg2qwerty_data  
  inflating: /content/emg2qwerty/data/emg2qwerty_data/2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5  
  inflating: /content/emg2qwerty/data/__MACOSX/emg2qwerty_data/._2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5  
  inflating: /content/emg2qwerty/data/emg2qwerty_data/2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5  
  inflating: /content/emg2qwerty/data/__MACOSX/emg2qwerty_data/._2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5  
  inflating: /content/emg2qwerty/data/emg2qwerty_data/2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5  
  inflating: /content/emg2qwerty/data/__MACOSX/emg2qwerty_data/._2021-06-04-1622863166

### Step 4: Start your experiments!

- Remember to download and copy the dataset to this directory: `Your_Dir/emg2qwerty/data`.
- You may now start your experiments with any scripts! Below are examples of single-user training and testing (greedy decoding).
- **There are two ways to track the logs:**
  - 1. Keep `--multirun`, and the logs will not be printed here, but they will be saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/submitit_logs/`.
  - 2. Comment out `--multirun` and the logs will be printed in this notebook, but they will not be saved.

#### Training

- The checkpoints are saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/checkpoints/`.

In [9]:
# Single-user training
!python -m emg2qwerty.train \
  user="single_user" \
  model=transformer \
  trainer.accelerator=gpu trainer.devices=1 \
  # --multirun

[2026-03-05 23:35:15,242][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

#### Testing:

- Replace `Your_Path_to_Checkpoint` with your checkpoint path.

In [10]:
# Single-user testing
!python -m emg2qwerty.train \
  user="single_user" \
  model=transformer \
  'checkpoint="/content/emg2qwerty/logs/2026-03-05/23-35-15/checkpoints/epoch=39-step=4800.ckpt"' \
  train=False trainer.accelerator=gpu \
  decoder=ctc_greedy \
  hydra.launcher.mem_gb=64 \
  # --multirun

[2026-03-06 00:38:44,084][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f